In [1]:
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.variant_index import VariantIndex
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment
from pyspark.sql import functions as f


Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



In [2]:
session = Session(extended_spark_conf={"spark.driver.memory": "10g"})


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/04 13:57:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/04 13:57:33 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/12/04 13:57:33 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/12/04 13:57:33 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/12/04 13:57:33 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


In [3]:
path_to_release_folder = "../../data/25.06/"
path_to_intermediate_data_folder = "../../data/intermediate_files/"

sl = StudyLocus.from_parquet(session, path_to_release_folder + "output/credible_set")
si = StudyIndex.from_parquet(session, path_to_release_folder + "output/study")

sl_eff = session.spark.read.parquet(path_to_intermediate_data_folder + "lead_variant_effect")


In [4]:
output_path = "../../data/intermediate_files/"


# Original data set


In [ ]:
# combinig it with l2g predictions
l2g = session.spark.read.parquet(path_to_release_folder + "irene_1208_l2g_predictions").select(
    "studyLocusId", "geneId", "score"
)


In [ ]:
fm = session.spark.read.parquet(path_to_release_folder + "/output/l2g_feature_matrix")
fm = fm.filter(f.col("isProteinCoding") == 1).cache()
fm.count()


25/12/04 13:26:51 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


10623371

In [ ]:
combined_df = (
    fm.join(l2g.select("studyLocusId", "geneId", "score"), on=["geneId", "studyLocusId"], how="left").fillna(0).cache()
)
combined_df.count()


10623371

In [ ]:
clpp_thr = 0.01
coloc_thr = 0.8

combined_df = combined_df.withColumn(
    "eQTL_coloc",
    f.when((f.col("eQtlColocClppMaximum") >= clpp_thr) | (f.col("eQtlColocH4Maximum") >= coloc_thr), 1).otherwise(0),
)
combined_df = combined_df.withColumn(
    "pQTL_coloc",
    f.when((f.col("pQtlColocClppMaximum") >= clpp_thr) | (f.col("pQtlColocH4Maximum") >= coloc_thr), 1).otherwise(0),
)
combined_df = combined_df.withColumn("VEP", f.when((f.col("vepMaximum") >= 0.66), 1).otherwise(0))
# combined_df = combined_df.withColumn(
#    "distance",
#    f.when((f.col("distanceSentinelFootprintNeighbourhood")==1) |
#    (f.col("distanceSentinelTssNeighbourhood")==1), 1).otherwise(0)
# ).cache()

combined_df = combined_df.withColumn(
    "distanceTSS", f.when(f.col("distanceSentinelTssNeighbourhood") == 1, 1).otherwise(0)
).cache()


combined_df.count()


10623371

In [ ]:
l2g_05 = l2g.filter(f.col("score") >= 0.5).cache()
cs_with_l2g = l2g_05.select("studyLocusId").distinct()
cs_with_l2g.count()


480623

In [ ]:
l2g_05.count()


503497

In [ ]:
l2g_no_05 = l2g.join(cs_with_l2g, on="studyLocusId", how="left_anti").cache()
l2g_no_05.count()


688918

In [ ]:
l2g_no_05.select("studyLocusId").distinct().count()


292355

In [ ]:
from pyspark.sql import Window

# Define a window specification partitioned by studyLocusId and ordered by score descending
window_spec = Window.partitionBy("studyLocusId").orderBy(f.desc("score"))

# Add a row_number column to rank rows within each studyLocusId partition
l2g_no_05_ranked = l2g_no_05.withColumn("row_number", f.row_number().over(window_spec))

# Filter rows where row_number is 1 (i.e., the max score for each studyLocusId)
l2g_no_05_max = l2g_no_05_ranked.filter(f.col("row_number") == 1).drop("row_number")

# Show the result
l2g_no_05_max.count()


292355

In [ ]:
l2g_no_05_max.select("studyLocusId").distinct().count()


292355

In [ ]:
l2g_no_05_max.show(4)


+--------------------+---------------+-------------------+
|        studyLocusId|         geneId|              score|
+--------------------+---------------+-------------------+
|002462a2da2f7c279...|ENSG00000215547|0.36747118830680847|
|00a70f45252881a9f...|ENSG00000254636| 0.3146820664405823|
|00badf4cd2ff71a2c...|ENSG00000105655| 0.2848552167415619|
|00bbe340d79fe1aa1...|ENSG00000077549|  0.485416442155838|
+--------------------+---------------+-------------------+
only showing top 4 rows



In [ ]:
l2g_no_05_max_01 = l2g_no_05_max.filter(f.col("score") >= 0.1).cache()
l2g_no_05_max_01.count()


285270

In [ ]:
l2g_prioritised = l2g_no_05_max_01.unionByName(l2g_05.select("studyLocusId", "geneId", "score")).cache()
l2g_prioritised.count()


788767

In [ ]:
261020 + 524241


785261

In [ ]:
l2g_prioritised.show(1)


+--------------------+---------------+-------------------+
|        studyLocusId|         geneId|              score|
+--------------------+---------------+-------------------+
|002462a2da2f7c279...|ENSG00000215547|0.36747118830680847|
+--------------------+---------------+-------------------+
only showing top 1 row



In [ ]:
final = (
    combined_df.select("studyLocusId", "geneId", "score", "eQTL_coloc", "pQTL_coloc", "VEP", "distanceTSS")
    .join(l2g_prioritised.drop("score"), on=["studyLocusId", "geneId"], how="inner")
    .cache()
)
final.count()


788767

In [ ]:
final.show(2)


+--------------------+---------------+------------------+----------+----------+---+-----------+
|        studyLocusId|         geneId|             score|eQTL_coloc|pQTL_coloc|VEP|distanceTSS|
+--------------------+---------------+------------------+----------+----------+---+-----------+
|2ef4eb65d7f430a75...|ENSG00000000971|0.8191224336624146|         0|         0|  0|          1|
|469a4c2b6f1247cbe...|ENSG00000000971|0.6837719082832336|         1|         0|  1|          1|
+--------------------+---------------+------------------+----------+----------+---+-----------+
only showing top 2 rows



In [ ]:
final.select("studyLocusId").distinct().count()


765893

In [ ]:
final.write.mode("overwrite").parquet(output_path + "list_of_prioritised_genes_per_CS.parquet")


# Data set with MAF and year


In [5]:
l2g_full = session.spark.read.parquet(path_to_intermediate_data_folder + "list_of_prioritised_genes_per_CS.parquet")


In [6]:
sl_eff = session.spark.read.parquet(path_to_intermediate_data_folder + "lead_variant_effect")


In [7]:
si = si.df.withColumn(
    "publicationDate",
    f.when(f.col("projectId") == "FINNGEN_R12", f.lit("2024-11-04")).otherwise(f.col("publicationDate")),
).withColumn("year", f.col("publicationDate").substr(1, 4).cast("int"))


In [8]:
si = si.withColumn(
    "is_nfe",
    f.when(
        f.size(
            f.filter(
                f.col("ldPopulationStructure"),
                lambda x: (x["ldPopulation"] == "nfe") & (x["relativeSampleSize"] >= 0.9),
            )
        )
        > 0,
        1,
    ).otherwise(0),
).cache()
si.count()


25/12/04 13:57:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


1966178

In [9]:
l2g_full.count()


788767

In [10]:
sl_eff.show(1)


+---------------+--------------------+--------------------+--------------------+---------------+----------+------------+---------------------+--------------------+-----------------+----------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------------------+------------------------+
|      variantId|             variant|        studyLocusId|             studyId|         geneId|diseaseIds|originalBeta|originalStandardError|     locusStatistics|finemappingMethod|isTransQtl|       variantEffect|majorLdPopulation|majorLdPopulationMaf| majorLdPopulationAf|   variantStatistics|     studyStatistics|  rescaledStatistics|leadVariantConsequence|traitFromSourceMappedIds|
+---------------+--------------------+--------------------+--------------------+---------------+----------+------------+---------------------+--------------------+-----------------+----------+--------------------+-----------------

In [11]:
# sl_eff.printSchema()


In [12]:
sl_eff = (
    sl_eff.drop("leadVariantConsequence")
    .withColumn("absBeta", f.abs(f.col("rescaledStatistics.absEstimatedBeta")))
    .withColumn("maf", f.col("majorLdPopulationMaf.value"))
)


In [13]:
sl_eff.show(1)


+---------------+--------------------+--------------------+--------------------+---------------+----------+------------+---------------------+--------------------+-----------------+----------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------------+------------------+------------------+
|      variantId|             variant|        studyLocusId|             studyId|         geneId|diseaseIds|originalBeta|originalStandardError|     locusStatistics|finemappingMethod|isTransQtl|       variantEffect|majorLdPopulation|majorLdPopulationMaf| majorLdPopulationAf|   variantStatistics|     studyStatistics|  rescaledStatistics|traitFromSourceMappedIds|           absBeta|               maf|
+---------------+--------------------+--------------------+--------------------+---------------+----------+------------+---------------------+--------------------+-----------------+----------+--------

In [14]:
l2g_full = l2g_full.join(
    sl_eff.select("studyId", "studyLocusId", "maf", "variantId", "absBeta"), on="studyLocusId", how="inner"
).cache()
l2g_full.count()


788767

In [15]:
l2g_full.show(1)


+--------------------+---------------+------------------+----------+----------+---+-----------+------------+-------------------+---------------+-------------------+
|        studyLocusId|         geneId|             score|eQTL_coloc|pQTL_coloc|VEP|distanceTSS|     studyId|                maf|      variantId|            absBeta|
+--------------------+---------------+------------------+----------+----------+---+-----------+------------+-------------------+---------------+-------------------+
|0005218bc3a62e387...|ENSG00000099337|0.8467043042182922|         0|         0|  1|          1|GCST90014010|0.02581275253566422|19_38320104_C_G|0.03758560384464141|
+--------------------+---------------+------------------+----------+----------+---+-----------+------------+-------------------+---------------+-------------------+
only showing top 1 row



In [16]:
l2g_full = l2g_full.join(si.select("studyId", "year", "is_nfe", "diseaseIds"), on="studyId", how="inner").cache()
l2g_full.count()


788767

In [17]:
l2g_full = (
    l2g_full.fillna({"maf": 0})
    .withColumn("nfe_common", f.when((f.col("is_nfe") == 1) & (f.col("maf") >= 0.01), 1).otherwise(0))
    .withColumn("non_nfe_common", f.when((f.col("is_nfe") == 0) & (f.col("maf") >= 0.01), 1).otherwise(0))
    .withColumn("rare", f.when((f.col("maf") < 0.01), 1).otherwise(0))
    .cache()
)
l2g_full.count()


788767

In [18]:
l2g_full.show(1)


+--------------------+--------------------+---------------+------------------+----------+----------+---+-----------+-------------------+---------------+-------------------+----+------+-------------+----------+--------------+----+
|             studyId|        studyLocusId|         geneId|             score|eQTL_coloc|pQTL_coloc|VEP|distanceTSS|                maf|      variantId|            absBeta|year|is_nfe|   diseaseIds|nfe_common|non_nfe_common|rare|
+--------------------+--------------------+---------------+------------------+----------+----------+---+-----------+-------------------+---------------+-------------------+----+------+-------------+----------+--------------+----+
|FINNGEN_R12_AUTOI...|c9c6c84efb4a00b17...|ENSG00000163599|0.7510585188865662|         1|         0|  0|          1|0.35754931714719274|2_203880280_T_C|0.28514203430891577|2024|     0|[EFO_0004237]|         0|             1|   0|
+--------------------+--------------------+---------------+------------------+--

In [19]:
l2g_full.write.mode("overwrite").parquet(
    path_to_intermediate_data_folder + "list_of_prioritised_genes_per_CS_with_year_nfe_maf.parquet"
)


# Data set for enrichemnt


In [20]:
l2g_full = session.spark.read.parquet(
    path_to_intermediate_data_folder + "list_of_prioritised_genes_per_CS_with_year_nfe_maf.parquet"
)


In [21]:
qd_cs = (
    session.spark.read.parquet(path_to_intermediate_data_folder + "qualifying_credible_sets")
    .select("studyLocusId")
    .cache()
)
qd_cs.count()


70618

In [22]:
l2g_full = l2g_full.join(qd_cs, "studyLocusId", "inner").cache()
l2g_full.count()


70400

In [23]:
l2g_full.select("geneId").distinct().count()


8285

In [24]:
l2g_full.write.mode("overwrite").parquet(path_to_intermediate_data_folder + "l2g_full_for_enrichment")
